# Sprint 4 — Target Construction & Graph Traversal

Spec: [`sprint-4-tasks.md`](../../litemapper/docs/requirements/sprint-4-tasks.md) — 11 tasks (S4-T00..T10) that shipped target-instance construction + recursive graph traversal.

| Scope                                           | Task    |
| ----------------------------------------------- | ------- |
| Parameterless constructor                       | S4-T01  |
| Best-match constructor                          | S4-T02  |
| Record / primary ctor / `init`-only             | S4-T03  |
| Null-safe navigation                            | S4-T05  |
| Circular reference tracking                     | S4-T06  |
| Nested object recursive mapping                 | S4-T07  |
| Depth-limit enforcement                         | S4-T08  |
| `BlueprintCompiler` + `MappingDelegateCache`    | S4-T09  |


## Setup


In [ ]:
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
Console.WriteLine("Ready.");


## 1. Records + `init`-only properties

The construction resolver picks the best constructor by matching argument names to the origin's member graph. Records and positional-constructor types are supported without any extra configuration.


In [ ]:
public sealed class UserEntity { public int Id { get; set; } public string Name { get; set; } = ""; public string Email { get; set; } = ""; }

// Target: positional record (primary constructor + init-only props).
public sealed record UserRecord(int Id, string Name, string Email);

// Target: init-only class.
public sealed class UserInitOnly
{
    public int    Id    { get; init; }
    public string Name  { get; init; } = "";
    public string Email { get; init; } = "";
}

var sculptor = new SculptorBuilder()
    .Configure(o =>
    {
        o.Bind<UserEntity, UserRecord>(_ => { });
        o.Bind<UserEntity, UserInitOnly>(_ => { });
    })
    .Forge();

var src = new UserEntity { Id = 1, Name = "Ada", Email = "ada@example.com" };

var rec = sculptor.Map<UserEntity, UserRecord>(src);
Console.WriteLine($"Record   : {rec}");

var init = sculptor.Map<UserEntity, UserInitOnly>(src);
Console.WriteLine($"InitOnly : Id={init.Id}, Name={init.Name}, Email={init.Email}");


## 2. `BuildWith(factory)` — explicit construction expression

When the convention-picked constructor is wrong (multiple ambiguous matches, or domain invariants the mapper can't infer), supply an explicit factory expression. The expression is compiled into the mapping delegate — no reflection at mapping time.


In [ ]:
public sealed class InvoiceEntity { public int Id { get; init; } public decimal Net { get; init; } public decimal Tax { get; init; } }

public sealed class InvoiceView
{
    public InvoiceView(int id, decimal gross) { Id = id; Gross = gross; }
    public int     Id    { get; }
    public decimal Gross { get; }
}

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<InvoiceEntity, InvoiceView>(rule => rule
        .BuildWith(e => new InvoiceView(e.Id, e.Net + e.Tax))))
    .Forge();

var view = sculptor.Map<InvoiceEntity, InvoiceView>(new InvoiceEntity { Id = 7, Net = 100m, Tax = 10m });
Console.WriteLine($"InvoiceView(Id={view.Id}, Gross={view.Gross:0.00})");


## 3. Nested recursive mapping

Any registered target member whose type has its own `Bind<>` pair is mapped recursively. No explicit per-property rule is needed.


In [ ]:
public sealed class Addr          { public string City { get; init; } = ""; public string Zip { get; init; } = ""; }
public sealed class AddrDto       { public string City { get; set; } = ""; public string Zip { get; set; } = ""; }
public sealed class CustomerFull  { public string Name { get; init; } = ""; public Addr Address { get; init; } = new(); }
public sealed class CustomerFullDto { public string Name { get; set; } = ""; public AddrDto Address { get; set; } = new(); }

var sculptor = new SculptorBuilder()
    .Configure(o =>
    {
        o.Bind<Addr, AddrDto>(_ => { });
        o.Bind<CustomerFull, CustomerFullDto>(_ => { });
    })
    .Forge();

var src = new CustomerFull { Name = "Ada", Address = new Addr { City = "London", Zip = "EC1" } };
var dto = sculptor.Map<CustomerFull, CustomerFullDto>(src);
Console.WriteLine($"Name = {dto.Name}");
Console.WriteLine($"Address.City = {dto.Address.City}");
Console.WriteLine($"Address.Zip  = {dto.Address.Zip}");


## 4. `TrackReferences()` — circular-graph safe mapping

Without tracking, a cyclic graph (`A → B → A`) would recurse until the stack overflows. `.TrackReferences()` wires an identity map into the scope so every origin instance is mapped at most once; subsequent visits return the already-mapped target.


In [ ]:
public sealed class Node    { public int Id { get; init; } public Node? Next { get; set; } }
public sealed class NodeDto { public int Id { get; set; } public NodeDto? Next { get; set; } }

// Build a tiny cycle: 1 → 2 → 1.
var a = new Node { Id = 1 };
var b = new Node { Id = 2 };
a.Next = b;
b.Next = a;

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<Node, NodeDto>(rule => rule.TrackReferences()))
    .Forge();

var dto = sculptor.Map<Node, NodeDto>(a);
Console.WriteLine($"dto.Id            = {dto.Id}");
Console.WriteLine($"dto.Next.Id       = {dto.Next!.Id}");
Console.WriteLine($"dto.Next.Next.Id  = {dto.Next.Next!.Id}  (cycle preserved)");
Console.WriteLine($"Same ref: dto == dto.Next.Next?  {object.ReferenceEquals(dto, dto.Next.Next)}");


## 5. `DepthLimit(n)` — clamp a recursive graph

Cap the depth at which nested mapping stops. Beyond the limit, target references are left `null` instead of continuing to recurse.


In [ ]:
public sealed class Comment    { public string Text { get; init; } = ""; public Comment? Reply { get; set; } }
public sealed class CommentDto { public string Text { get; set; } = ""; public CommentDto? Reply { get; set; } }

// 4-deep comment chain.
var chain = new Comment { Text = "root", Reply = new Comment { Text = "r1", Reply = new Comment { Text = "r2", Reply = new Comment { Text = "r3" } } } };

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<Comment, CommentDto>(rule => rule.DepthLimit(2)))
    .Forge();

var dto = sculptor.Map<Comment, CommentDto>(chain);
Console.WriteLine($"depth 0: {dto.Text}");
Console.WriteLine($"depth 1: {dto.Reply?.Text}");
Console.WriteLine($"depth 2: {dto.Reply?.Reply?.Text}");
Console.WriteLine($"depth 3: {(dto.Reply?.Reply?.Reply?.Text ?? "<null — clamped>")}");


## Next

- **`sprint-05-collections.ipynb`** — how the mapper dispatches over arrays, lists, sets, dictionaries, and immutables.
